# Scenario 1 — FAQ Agent with Simple RAG

## Objective
Create a FAQ agent for **TechMart** (online electronics store) that:
1. Receives a customer question
2. **Classifies** the question type automatically
3. **Searches** the correct knowledge base for an answer (RAG / FAISS)
4. **Generates** a contextualized response with the LLM

## LangGraph Concepts Explored

| Concept | Description |
|---|---|
| **State** | Typed dictionary that carries data between nodes |
| **Node** | Function that receives the state and returns updated fields |
| **Edge** | Fixed connection between two nodes |
| **Conditional Edge** | Dynamic routing based on state |

## Flow Architecture

> **Two knowledge bases** (FAISS):
> - `policies` — exchanges, shipping, payment, warranty, TechPoints
> - `products` — specifications, prices, product recommendations


## Step 1 — Imports and Initial Setup


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from shared.llm import get_llm
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_databricks import DatabricksEmbeddings
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

print("Dependencies loaded successfully!")


## Step 2 — Creating the Vector Databases with FAISS

Let's load the two TechMart Markdown files and create local vector indexes.

The process is:
1. **Load** the `.md` file with `TextLoader`
2. **Split** the text into smaller pieces (chunks) with `RecursiveCharacterTextSplitter`
3. **Generate embeddings** for each chunk with `DatabricksEmbeddings`
4. **Index** the embeddings in a local FAISS vector store


In [ ]:
def create_vector_store(file_path):
    """Loads a .md file and creates a FAISS vector store."""

    loader = TextLoader(file_path, encoding="utf-8")
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=50,
        separators=["\n## ", "\n### ", "\n\n", "\n", " "],
    )
    chunks = splitter.split_documents(documents)

    embeddings = DatabricksEmbeddings(endpoint=os.getenv("DATABRICKS_EMBEDDINGS_ENDPOINT", "databricks-bge-large-en"))
    store = FAISS.from_documents(chunks, embeddings)

    print(f"Vector store created: {file_path}")
    print(f"   -> {len(chunks)} chunks indexed")
    return store


policies_store = create_vector_store("data/kb/store_policies.md")
products_store = create_vector_store("data/kb/product_faq.md")

print("\nVector stores ready for use!")


## Step 3 — Defining the State

The **State** is the heart of a LangGraph graph. It defines **which information flows** between nodes.

Each node:
- **Receives** the current state as a parameter
- **Returns** a dictionary with only the fields it wants to update

LangGraph handles the merge automatically.


In [ ]:
class FAQState(TypedDict):
    """FAQ agent state.

    Fields:
        question  — original customer question
        category  — classification: 'policies' or 'products'
        context   — relevant chunks retrieved via RAG
        response  — final response generated by the LLM
    """
    question: str
    category: str
    context: str
    response: str

print("State defined — FAQState with 4 fields")


## Step 4 — Creating the Node Functions

We will create **4 nodes** (functions):

| Node | Role |
|---|---|
| `classify_question` | Uses the LLM to decide if the question is about *policies* or *products* |
| `search_policies` | Searches for similarity in the FAISS policies store |
| `search_products` | Searches for similarity in the FAISS products store |
| `generate_response` | Generates the final response using the retrieved context |


In [ ]:
llm = get_llm()


def classify_question(state: FAQState) -> dict:
    """Classifies the question as 'policies' or 'products'."""

    question = state["question"]
    print(f"\nClassifying: '{question}'")

    prompt = f"""Classify the following question from a TechMart customer (electronics store).
Reply with ONLY one word — no quotes, no punctuation:
- policies  -> returns, exchanges, shipping, payment, warranty, TechPoints, hours
- products  -> specifications, recommendations, prices, models, compatibility

Question: {question}
Category:"""

    resp = llm.invoke(prompt)
    category = resp.content.strip().lower().replace('\"', "").replace("'", "")

    if category not in ("policies", "products"):
        category = "products"

    print(f"   Category -> {category}")
    return {"category": category}


def search_policies(state: FAQState) -> dict:
    """Retrieves relevant chunks from the policies knowledge base."""

    print("   Searching the POLICIES store...")
    docs = policies_store.similarity_search(state["question"], k=3)
    context = "\n\n".join(d.page_content for d in docs)
    print(f"   {len(docs)} chunks found")
    return {"context": context}


def search_products(state: FAQState) -> dict:
    """Retrieves relevant chunks from the product FAQ store."""

    print("   Searching the PRODUCTS store...")
    docs = products_store.similarity_search(state["question"], k=3)
    context = "\n\n".join(d.page_content for d in docs)
    print(f"   {len(docs)} chunks found")
    return {"context": context}


def generate_response(state: FAQState) -> dict:
    """Generates the final response based on the retrieved context."""

    print("   Generating response...")

    prompt = f"""You are a virtual assistant for TechMart, an online electronics store.
Respond in a clear, polite, and helpful manner.
Use ONLY the information from the context below. If you don't know, say you couldn't find the answer.

Context:
{state['context']}

Customer question: {state['question']}

Response:"""

    resp = llm.invoke(prompt)
    print("   Response generated!")
    return {"response": resp.content}


print("4 node functions created")


## Step 5 — Building the LangGraph Graph

Now we connect everything:

1. `START` -> `classify`
2. `classify` -> **conditional routing** (policies *or* products)
3. each search -> `respond`
4. `respond` -> `END`


In [ ]:
def route_by_category(state: FAQState) -> str:
    """Routing function — returns the name of the next node."""
    return state["category"]

graph = StateGraph(FAQState)

graph.add_node("classify", classify_question)
graph.add_node("search_policies", search_policies)
graph.add_node("search_products", search_products)
graph.add_node("respond", generate_response)

graph.add_edge(START, "classify")
graph.add_edge("search_policies", "respond")
graph.add_edge("search_products", "respond")
graph.add_edge("respond", END)

graph.add_conditional_edges(
    "classify",
    route_by_category,
    {
        "policies": "search_policies",
        "products": "search_products",
    },
)

app = graph.compile()


In [ ]:
from IPython.display import Image, display

print("Graph compiled!\n")
print("Mermaid diagram of the graph:")
display(Image(app.get_graph().draw_mermaid_png()))


## Step 6 — Testing the Agent

Let's send different questions and observe:
- Which **category** the classifier chooses
- Which **vector store** is queried
- The **final response** generated


In [ ]:
test_questions = [
    "What is the deadline to return a product?",
    "Which notebook do you recommend for programming?",
    "Do you accept wire transfers? Is there a discount?",
    "Which monitors do you sell?",
    "How does the TechPoints program work?",
]

for question in test_questions:
    print("=" * 70)
    result = app.invoke({"question": question})
    print(f"\nFINAL RESPONSE:\n{result['response']}")
    print("=" * 70)
    print()


## Step 7 — Interactive Test (optional)

Run the cell below to test your own questions.


In [ ]:
# Uncomment to test interactively:
# question = input("Enter your question: ")
# result = app.invoke({"question": question})
# print(f"\n{result['response']}")


## Exercises

1. **New category**: Create a third `.md` file (e.g., `data/kb/new_kb.md`) and add a third route to the graph.

2. **Improve the classifier**: Instead of asking the LLM to classify, use embedding similarity with each store to choose the most relevant one.

3. **Validation node**: Add a node after `respond` that checks whether the response actually answers the question (and requests a rewrite if needed).

---

## Summary

| LangGraph Concept | What We Used |
|---|---|
| `StateGraph` | Created the graph with `FAQState` |
| `add_node` | Added 4 nodes (Python functions) |
| `add_edge` | Fixed connections (search -> respond -> END) |
| `add_conditional_edges` | Dynamic routing (classify -> policies/products) |
| `compile` | Compiled the graph into an executable app |
| `invoke` | Executed the graph by passing the initial state |
